**Задание**

На вебинаре мы говорили, что долгое время CNN и RNN архитектуры были конурируещими. Выяснить, какая архитектура больше подходит для задачи сантимент анализа на данных с вебинара:
1. построить свёрточные архитектуры;
2. построить различные архитектуры с RNN;
3. построить совместные архитектуры CNN -> RNN  и (RNN -> CNN);
4. сдлать выводы что получилось лучше.

In [1]:
# Подключаем диск
from google.colab import drive
drive.mount('./drive')

Mounted at ./drive


In [2]:
!pip -q install pymorphy2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 72.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
import numpy as np
import pandas as pd

import nltk
from nltk.probability import FreqDist
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from pymorphy2 import MorphAnalyzer

import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from keras.utils import to_categorical
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Input, Embedding, Conv1D, GlobalMaxPool1D, Flatten, SimpleRNN, LSTM, GRU, Reshape
from keras.losses import categorical_crossentropy, SparseCategoricalCrossentropy

In [4]:
data = pd.read_excel('/content/drive/MyDrive/NLP/отзывы за лето.xls')
data.head()

,Rating,Content,Date
0,5,It just works!,2017-08-14
1,4,В целом удобноное приложение...из минусов хотя...,2017-08-14
2,5,Отлично все,2017-08-14
3,5,Стал зависать на 1% работы антивируса. Дальше ...,2017-08-14
4,5,"Очень удобно, работает быстро.",2017-08-14


In [5]:
max_words = 20000
max_len = 150
num_classes = 5

epochs = 20
batch_size = 512
print_batch_n = 100

In [6]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [7]:
stopwordslist = stopwords.words('russian')
ptrn = r'[^a-zA-Zа-яА-Я0-9]'

morpher = MorphAnalyzer()

def words_only(text):
    text=str(text)
    return text.lower()

def remove_punkt(text):
    return re.sub(ptrn, ' ', text)

def to_token(text):
    return nltk.tokenize.word_tokenize(text)

def remove_stopwords(text):
    text_list = [w for w in text if w not in stopwordslist]
    return ' '.join(word for word in text_list)

def morphe_text(text):
    text = [morpher.parse(word)[0].normal_form for word in text.split() if word not in stopwordslist]
    return " ".join(text)

def normalize_text(text):
    text = words_only(text)
    text = remove_punkt(text)
    text = to_token(text)
    text = remove_stopwords(text)
    text = morphe_text(text)
    return text

In [8]:
data['normalized_content'] = data['Content'].apply(normalize_text)
data.head()

,Rating,Content,Date,normalized_content
0,5,It just works!,2017-08-14,it just works
1,4,В целом удобноное приложение...из минусов хотя...,2017-08-14,целое удобноной приложение минус хотеть слишко...
2,5,Отлично все,2017-08-14,отлично
3,5,Стал зависать на 1% работы антивируса. Дальше ...,2017-08-14,стать зависать 1 работа антивирус далёкий нику...
4,5,"Очень удобно, работает быстро.",2017-08-14,очень удобно работать быстро


In [9]:
train_corpus = ' '.join(data['normalized_content'])
train_tokens = word_tokenize(train_corpus)
train_tokens_filtered = [word for word in train_tokens if word.isalnum()]

In [10]:
dist = FreqDist(train_tokens_filtered)
tokens_filtered_top = [pair[0] for pair in dist.most_common(max_words-1)]
voc = {v: k for k, v in dict(enumerate(tokens_filtered_top, 1)).items()}

In [11]:
def text_to_sequence(text, maxlen):
    result = []
    tokens = word_tokenize(text.lower())
    tokens_filtered = [word for word in tokens if word.isalnum()]
    for word in tokens_filtered:
        if word in voc:
            result.append(voc[word])
    padding = [0]*(maxlen-len(result))
    return padding + result[-maxlen:]

data_train = np.asarray(
    [text_to_sequence(text, max_len) for text in data['normalized_content']],
    dtype=np.int32)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(data_train, data.Rating, test_size=0.3, random_state=1)

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

**CNN**

In [13]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len))
model.add(Conv1D(128, 3))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Conv1D(128, 3))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(GlobalMaxPool1D())
model.add(Flatten())
model.add(Dense(10))
model.add(Activation('relu'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 18s 245ms/step - loss: 1.1687 - accuracy: 0.6723 - val_loss: 1.1875 - val_accuracy: 0.7155
Epoch 2/5
23/23 [==============================] - 3s 135ms/step - loss: 0.8406 - accuracy: 0.7310 - val_loss: 0.8555 - val_accuracy: 0.7532
Epoch 3/5
23/23 [==============================] - 2s 102ms/step - loss: 0.7102 - accuracy: 0.7666 - val_loss: 0.7835 - val_accuracy: 0.7608
Epoch 4/5
23/23 [==============================] - 2s 95ms/step - loss: 0.6404 - accuracy: 0.7821 - val_loss: 0.7494 - val_accuracy: 0.7542
Epoch 5/5
23/23 [==============================] - 2s 75ms/step - loss: 0.5907 - accuracy: 0.7984 - val_loss: 0.7225 - val_accuracy: 0.7556


In [14]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = pd.DataFrame(columns=['model', 'Test score', 'Test accuracy'])

metrics_df = metrics_df.append({
    'model': 'CNN',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 0s 22ms/step - loss: 0.7579 - accuracy: 0.7339
Test score: 0.7578664422035217
Test accuracy: 0.7339464426040649


<ipython-input-14-f8505f291b9a>:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**SimpleRNN**

In [15]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len,
                    trainable=True,
                    mask_zero=True))
model.add(SimpleRNN(128))
model.add(Activation("relu"))
model.add(Dropout(0.2))
model.add(Reshape((1,128)))
model.add(SimpleRNN(128))
model.add(Activation("relu"))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(10))
model.add(Activation("relu"))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 11s 298ms/step - loss: 1.3869 - accuracy: 0.6391 - val_loss: 0.9786 - val_accuracy: 0.7155
Epoch 2/5
23/23 [==============================] - 10s 422ms/step - loss: 0.7956 - accuracy: 0.7311 - val_loss: 0.7167 - val_accuracy: 0.7598
Epoch 3/5
23/23 [==============================] - 6s 276ms/step - loss: 0.6664 - accuracy: 0.7785 - val_loss: 0.6666 - val_accuracy: 0.7791
Epoch 4/5
23/23 [==============================] - 9s 409ms/step - loss: 0.6104 - accuracy: 0.7913 - val_loss: 0.6599 - val_accuracy: 0.7829
Epoch 5/5
23/23 [==============================] - 6s 251ms/step - loss: 0.5676 - accuracy: 0.8052 - val_loss: 0.6722 - val_accuracy: 0.7819


In [16]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = metrics_df.append({
    'model': 'SimpleRNN',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 0s 18ms/step - loss: 0.6988 - accuracy: 0.7617
Test score: 0.6988376975059509
Test accuracy: 0.7616972923278809


<ipython-input-16-726e1a524894>:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**LSTM**

In [17]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len,
                    trainable=True,
                    mask_zero=True))
model.add(LSTM(128))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Reshape((1,128)))
model.add(LSTM(128))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(10))
model.add(Activation('relu'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 22s 606ms/step - loss: 1.5457 - accuracy: 0.6814 - val_loss: 1.3786 - val_accuracy: 0.7155
Epoch 2/5
23/23 [==============================] - 12s 510ms/step - loss: 1.0804 - accuracy: 0.7030 - val_loss: 0.8470 - val_accuracy: 0.7155
Epoch 3/5
23/23 [==============================] - 12s 544ms/step - loss: 0.7648 - accuracy: 0.7031 - val_loss: 0.7321 - val_accuracy: 0.7297
Epoch 4/5
23/23 [==============================] - 11s 492ms/step - loss: 0.6799 - accuracy: 0.7629 - val_loss: 0.6963 - val_accuracy: 0.7770
Epoch 5/5
23/23 [==============================] - 10s 434ms/step - loss: 0.6269 - accuracy: 0.7887 - val_loss: 0.6819 - val_accuracy: 0.7815


In [18]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = metrics_df.append({
    'model': 'LSTM',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 0s 34ms/step - loss: 0.7089 - accuracy: 0.7651
Test score: 0.7089344263076782
Test accuracy: 0.7650855183601379


<ipython-input-18-c7475f127a67>:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**GRU**

In [19]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len,
                    trainable=True,
                    mask_zero=True))
model.add(GRU(128))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Reshape((1,128)))
model.add(GRU(128))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(10))
model.add(Activation('relu'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 21s 637ms/step - loss: 1.4582 - accuracy: 0.6906 - val_loss: 1.1391 - val_accuracy: 0.7155
Epoch 2/5
23/23 [==============================] - 13s 588ms/step - loss: 0.9447 - accuracy: 0.7030 - val_loss: 0.8408 - val_accuracy: 0.7155
Epoch 3/5
23/23 [==============================] - 11s 480ms/step - loss: 0.7718 - accuracy: 0.7030 - val_loss: 0.7530 - val_accuracy: 0.7155
Epoch 4/5
23/23 [==============================] - 12s 492ms/step - loss: 0.6848 - accuracy: 0.7497 - val_loss: 0.7107 - val_accuracy: 0.7736
Epoch 5/5
23/23 [==============================] - 13s 567ms/step - loss: 0.6321 - accuracy: 0.7824 - val_loss: 0.6919 - val_accuracy: 0.7805


In [20]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = metrics_df.append({
    'model': 'GRU',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 1s 38ms/step - loss: 0.7178 - accuracy: 0.7628
Test score: 0.7178024053573608
Test accuracy: 0.7628267407417297


<ipython-input-20-c7a60f594fef>:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**CNN -> LSTM**

In [21]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len,
                    trainable=True,
                    mask_zero=True))
model.add(Conv1D(128, 3))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(LSTM(128))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(10))
model.add(Activation('relu'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 8s 193ms/step - loss: 1.2108 - accuracy: 0.6746 - val_loss: 0.9535 - val_accuracy: 0.7155
Epoch 2/5
23/23 [==============================] - 3s 109ms/step - loss: 0.8671 - accuracy: 0.7031 - val_loss: 0.7634 - val_accuracy: 0.7155
Epoch 3/5
23/23 [==============================] - 2s 103ms/step - loss: 0.7375 - accuracy: 0.7399 - val_loss: 0.7049 - val_accuracy: 0.7653
Epoch 4/5
23/23 [==============================] - 3s 109ms/step - loss: 0.6786 - accuracy: 0.7725 - val_loss: 0.6866 - val_accuracy: 0.7708
Epoch 5/5
23/23 [==============================] - 4s 175ms/step - loss: 0.6392 - accuracy: 0.7719 - val_loss: 0.6793 - val_accuracy: 0.7567


In [22]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = metrics_df.append({
    'model': 'CNN -> LSTM',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 0s 22ms/step - loss: 0.7069 - accuracy: 0.7490
Test score: 0.7068835496902466
Test accuracy: 0.748951256275177


<ipython-input-22-9fb914ee426c>:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**CNN -> GRU**

In [23]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len,
                    trainable=True,
                    mask_zero=True))
model.add(Conv1D(128, 3))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(GRU(128))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(10))
model.add(Activation('relu'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 7s 158ms/step - loss: 1.1602 - accuracy: 0.6786 - val_loss: 0.8125 - val_accuracy: 0.7155
Epoch 2/5
23/23 [==============================] - 2s 103ms/step - loss: 0.7713 - accuracy: 0.7245 - val_loss: 0.7068 - val_accuracy: 0.7542
Epoch 3/5
23/23 [==============================] - 2s 100ms/step - loss: 0.6765 - accuracy: 0.7656 - val_loss: 0.6712 - val_accuracy: 0.7670
Epoch 4/5
23/23 [==============================] - 2s 104ms/step - loss: 0.6107 - accuracy: 0.7822 - val_loss: 0.6664 - val_accuracy: 0.7708
Epoch 5/5
23/23 [==============================] - 3s 145ms/step - loss: 0.5528 - accuracy: 0.7997 - val_loss: 0.6896 - val_accuracy: 0.7712


In [24]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = metrics_df.append({
    'model': 'CNN -> GRU',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 0s 18ms/step - loss: 0.7160 - accuracy: 0.7544
Test score: 0.7159863114356995
Test accuracy: 0.7544369101524353


<ipython-input-24-3cab87a72cc7>:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**CNN -> SimpleRNN**

In [25]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len,
                    trainable=True,
                    mask_zero=True))
model.add(Conv1D(128, 3))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(SimpleRNN(128))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(10))
model.add(Activation('relu'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 12s 452ms/step - loss: 1.0913 - accuracy: 0.6688 - val_loss: 0.8574 - val_accuracy: 0.7176
Epoch 2/5
23/23 [==============================] - 5s 210ms/step - loss: 0.8078 - accuracy: 0.7048 - val_loss: 0.7321 - val_accuracy: 0.7172
Epoch 3/5
23/23 [==============================] - 5s 200ms/step - loss: 0.7081 - accuracy: 0.7180 - val_loss: 0.6823 - val_accuracy: 0.7335
Epoch 4/5
23/23 [==============================] - 7s 321ms/step - loss: 0.6497 - accuracy: 0.7394 - val_loss: 0.6693 - val_accuracy: 0.7463
Epoch 5/5
23/23 [==============================] - 4s 189ms/step - loss: 0.6127 - accuracy: 0.7560 - val_loss: 0.6687 - val_accuracy: 0.7729


In [26]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = metrics_df.append({
    'model': 'CNN -> SimpleRNN',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 0s 17ms/step - loss: 0.6981 - accuracy: 0.7590
Test score: 0.6980783343315125
Test accuracy: 0.7589545249938965


<ipython-input-26-13369e4be48f>:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**SimpleRNN -> CNN**

In [27]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len,
                    trainable=True,
                    mask_zero=True))
model.add(SimpleRNN(128, return_sequences=True))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Conv1D(128, 3))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(10))
model.add(Activation('relu'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 13s 433ms/step - loss: 1.1684 - accuracy: 0.7028 - val_loss: 0.8649 - val_accuracy: 0.7228
Epoch 2/5
23/23 [==============================] - 6s 271ms/step - loss: 0.7990 - accuracy: 0.7302 - val_loss: 0.7212 - val_accuracy: 0.7535
Epoch 3/5
23/23 [==============================] - 9s 402ms/step - loss: 0.6640 - accuracy: 0.7654 - val_loss: 0.6768 - val_accuracy: 0.7670
Epoch 4/5
23/23 [==============================] - 6s 273ms/step - loss: 0.5860 - accuracy: 0.7837 - val_loss: 0.6681 - val_accuracy: 0.7705
Epoch 5/5
23/23 [==============================] - 9s 400ms/step - loss: 0.5259 - accuracy: 0.7940 - val_loss: 0.6835 - val_accuracy: 0.7715


In [28]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = metrics_df.append({
    'model': 'SimpleRNN -> CNN',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 0s 22ms/step - loss: 0.7020 - accuracy: 0.7619
Test score: 0.7020390033721924
Test accuracy: 0.7618586421012878


<ipython-input-28-2d413a51352c>:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**LSTM -> CNN**

In [29]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len,
                    trainable=True,
                    mask_zero=True))
model.add(LSTM(128, return_sequences=True))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Conv1D(128, 3))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(10))
model.add(Activation('relu'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 19s 527ms/step - loss: 1.5120 - accuracy: 0.2886 - val_loss: 1.3802 - val_accuracy: 0.7155
Epoch 2/5
23/23 [==============================] - 12s 520ms/step - loss: 1.3025 - accuracy: 0.7031 - val_loss: 1.1987 - val_accuracy: 0.7155
Epoch 3/5
23/23 [==============================] - 12s 544ms/step - loss: 1.1083 - accuracy: 0.7030 - val_loss: 1.0896 - val_accuracy: 0.7155
Epoch 4/5
23/23 [==============================] - 12s 530ms/step - loss: 0.9634 - accuracy: 0.7030 - val_loss: 1.0222 - val_accuracy: 0.7155
Epoch 5/5
23/23 [==============================] - 10s 413ms/step - loss: 0.8778 - accuracy: 0.7030 - val_loss: 0.9796 - val_accuracy: 0.7155


In [30]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = metrics_df.append({
    'model': 'LSTM -> CNN',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 1s 38ms/step - loss: 0.9771 - accuracy: 0.7073
Test score: 0.9771266579627991
Test accuracy: 0.7073249220848083


<ipython-input-30-99076512101e>:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**GRU -> CNN**

In [31]:
model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_len,
                    trainable=True,
                    mask_zero=True))
model.add(GRU(128, return_sequences=True))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Conv1D(128, 3))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(10))
model.add(Activation('relu'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

model.compile(loss=SparseCategoricalCrossentropy(),
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=5,
          verbose=1,
          validation_split=0.2)

Epoch 1/5
23/23 [==============================] - 20s 666ms/step - loss: 1.3100 - accuracy: 0.6759 - val_loss: 0.9496 - val_accuracy: 0.7155
Epoch 2/5
23/23 [==============================] - 13s 574ms/step - loss: 0.8557 - accuracy: 0.7031 - val_loss: 0.7618 - val_accuracy: 0.7162
Epoch 3/5
23/23 [==============================] - 10s 454ms/step - loss: 0.6999 - accuracy: 0.7136 - val_loss: 0.6902 - val_accuracy: 0.7221
Epoch 4/5
23/23 [==============================] - 13s 541ms/step - loss: 0.6381 - accuracy: 0.7254 - val_loss: 0.6806 - val_accuracy: 0.7325
Epoch 5/5
23/23 [==============================] - 13s 554ms/step - loss: 0.5978 - accuracy: 0.7397 - val_loss: 0.6935 - val_accuracy: 0.7297


In [32]:
score = model.evaluate(X_test, y_test, batch_size=batch_size, verbose=1)

metrics_df = metrics_df.append({
    'model': 'GRU -> CNN',
    'Test score': score[0],
    'Test accuracy': score[1],
}, ignore_index=True)

print('Test score:', score[0])
print('Test accuracy:', score[1])

13/13 [==============================] - 0s 36ms/step - loss: 0.7053 - accuracy: 0.7228
Test score: 0.7053192853927612
Test accuracy: 0.7228137850761414


<ipython-input-32-1cb997e7e0ac>:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df = metrics_df.append({


**Итоги**

In [33]:
metrics_df.sort_values(by='Test score', ascending=False)

,model,Test score,Test accuracy
8,LSTM -> CNN,0.977127,0.707325
0,CNN,0.757866,0.733946
3,GRU,0.717802,0.762827
5,CNN -> GRU,0.715986,0.754437
2,LSTM,0.708934,0.765086
4,CNN -> LSTM,0.706884,0.748951
9,GRU -> CNN,0.705319,0.722814
7,SimpleRNN -> CNN,0.702039,0.761859
1,SimpleRNN,0.698838,0.761697
6,CNN -> SimpleRNN,0.698078,0.758955


In [34]:
metrics_df.sort_values(by='Test accuracy', ascending=False)

,model,Test score,Test accuracy
2,LSTM,0.708934,0.765086
3,GRU,0.717802,0.762827
7,SimpleRNN -> CNN,0.702039,0.761859
1,SimpleRNN,0.698838,0.761697
6,CNN -> SimpleRNN,0.698078,0.758955
5,CNN -> GRU,0.715986,0.754437
4,CNN -> LSTM,0.706884,0.748951
0,CNN,0.757866,0.733946
9,GRU -> CNN,0.705319,0.722814
8,LSTM -> CNN,0.977127,0.707325
